# Language modeling is one of the fundamental concepts behind AI systems such as ChatGPT, Gemini, and Claude. Since you are working with Harry Potter text and want to build your own language model in Python, let's understand it from that perspective.
1. What is language modeling?
Language modeling is the process of learning patterns in language so that a model can estimate the probability of a sequence of words or predict what token is likely to come next.

Imagine your training data contains this sentence:
Harry Potter lived with his aunt and uncle.

During training, a simple language model learns patterns such as:
- After Harry, the next word might be Potter.
- After Harry Potter, the next word might be lived.
- After Harry Potter lived with his, the next word might be aunt.
The model learns these patterns from its training data instead of needing you to manually program every sentence.

In [82]:
from collections import Counter
import nltk
import numpy as np
import torch
from nltk.tokenize import word_tokenize
from torch.utils.data import Dataset, DataLoader

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /home/mmhe/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/mmhe/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/mmhe/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [83]:
with open("harry_potter_series_pure_story_split_lines.txt", "r", encoding="utf-8") as f:
    hPRaw = f.read()
    hPLines = hPRaw.splitlines()

tokens = word_tokenize(hPRaw.lower())
len(tokens)

1422108

In [84]:
vocab = {"<UNK>": 0}

for token in Counter(tokens).keys():  # noqa: SIM118
    if token not in vocab:
        vocab[token] = len(vocab)

len(vocab)

22717

In [85]:
def textToIdx(sentence, vocab):
    index = []

    for token in word_tokenize(sentence.lower()):
        if token in vocab:
            index.append(vocab[token])
        else:
            index.append(vocab["<UNK>"])
    return index

In [86]:
textToIdx("Harry is a good boy!", vocab= vocab)

[403, 889, 47, 146, 97, 295]

In [87]:
sentenceInNumForm = []

for line in hPLines:
    sentenceInNumForm.append(textToIdx(sentence = line, vocab = vocab))
len(sentenceInNumForm), len(hPLines)

(89406, 89406)

In [88]:
trainingSequence = []

for sentence in sentenceInNumForm:
    for seq in range(1, len(sentence)):
        trainingSequence.append(sentence[:seq + 1])

len(trainingSequence)

1332771

In [89]:
maxLength = 0
for seq in trainingSequence:
    l = len(seq)
    if maxLength < l:
        maxLength = l
    
maxLength

69

In [90]:
paddedTrainingSequence = []

for seq in trainingSequence:
    paddedTrainingSequence.append(([0] * (maxLength - len(seq))) + seq)
len(paddedTrainingSequence)

1332771

In [91]:
paddedTrainingSequence = torch.tensor(paddedTrainingSequence, dtype= torch.long)

X = paddedTrainingSequence[:, :-1]
y = paddedTrainingSequence[:, -1]
len(X), len(y)

(1332771, 1332771)

In [92]:
X, y

(tensor([[  0,   0,   0,  ...,   0,   0,   1],
         [  0,   0,   0,  ...,   0,   1,   2],
         [  0,   0,   0,  ...,   1,   2,   3],
         ...,
         [  0,   0,   0,  ...,   0,   0, 171],
         [  0,   0,   0,  ...,   0, 171,  45],
         [  0,   0,   0,  ..., 171,  45, 579]]),
 tensor([  2,   3,   4,  ...,  45, 579,  23]))

In [93]:
class CusDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [94]:
trainDataset = CusDataset(X, y)
len(trainDataset)

1332771

In [95]:
trainLoader = DataLoader(trainDataset, batch_size= 32, shuffle= True)
len(trainLoader)

41650